# Structural-shift embedding progression
Load the saved MLP and GNN checkpoints, plot MMD over training,
and inspect source/target embedding trajectories.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
run_specs = [{"name": "mlp", "embeddings": "/home/bini/codes/GDA/KDD/SCGDA/__saved__/struct_shift/run_0217_215147/mlp_embeddings.npz", "history": "/home/bini/codes/GDA/KDD/SCGDA/__saved__/struct_shift/run_0217_215147/mlp_history.npz"}, {"name": "gnn", "embeddings": "/home/bini/codes/GDA/KDD/SCGDA/__saved__/struct_shift/run_0217_215147/gnn_embeddings.npz", "history": "/home/bini/codes/GDA/KDD/SCGDA/__saved__/struct_shift/run_0217_215147/gnn_history.npz"}]

results = {}
for spec in run_specs:
    emb = np.load(spec['embeddings'])
    hist = np.load(spec['history'])
    results[spec['name']] = (emb, hist)

print('Loaded:', ', '.join(results.keys()))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, (_, hist) in results.items():
    ax.plot(hist['epoch'], hist['mmd_loss'], marker='o', linewidth=1.3, label=f'{name.upper()}')
ax.set_title('MMD progression over training')
ax.set_xlabel('Epoch')
ax.set_ylabel('MMD')
ax.grid(alpha=0.3)
ax.legend()
plt.show()


In [ ]:
def project2d(x):
    x = x - x.mean(axis=0, keepdims=True)
    if x.shape[1] <= 2:
        return x[:, :2]
    _, _, vh = np.linalg.svd(x, full_matrices=False)
    return x @ vh[:2].T

for name, (emb, hist) in results.items():
    idx = np.linspace(0, len(emb['epochs']) - 1, num=min(6, len(emb['epochs'])), dtype=int)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    epochs = emb['epochs'].tolist()
    for i in idx:
        src = project2d(emb['source'][i])
        tgt = project2d(emb['target'][i])
        alpha = 0.25
        if i == idx[-1]:
            alpha = 1.0
        ax[0].scatter(src[:, 0], src[:, 1], c=emb['source_labels'], alpha=alpha, cmap='tab10', s=16)
        ax[1].scatter(tgt[:, 0], tgt[:, 1], c=emb['target_labels'], alpha=alpha, cmap='tab10', s=16)
    ax[0].set_title(f'{name.upper()} source epochs: ' + ', '.join(map(str, [epochs[i] for i in idx])) )
    ax[1].set_title(f'{name.upper()} target epochs: ' + ', '.join(map(str, [epochs[i] for i in idx])) )
    for a in ax:
        a.set_xticks([])
        a.set_yticks([])
    plt.suptitle(f'{name.upper()} embedding snapshots')
    plt.tight_layout()
    plt.show()
